# GNN Shift Prediction Workflow

Dieses Notebook ermöglicht:
1. **Training** eines heterogenen GNN-Modells zur Shift-Vorhersage
2. **Vorhersage** mit einem trainierten Modell
3. **Erklärung** von Vorhersagen mittels GNNExplainer

---

## Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Projektpfade einrichten
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
EXPLAINER_DIR = SCRIPTS_DIR / "explainer"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

# Zentrale In-Dim-Definition (Single Source of Truth)
IN_DIM_DICT = {"H": 33, "C": 39, "Others": 16}

# Pfade zum sys.path hinzufügen
for path in [str(SCRIPTS_DIR), str(PROJECT_ROOT), str(EXPLAINER_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Projekt-Root: {PROJECT_ROOT}")
print(f"Scripts-Ordner: {SCRIPTS_DIR}")
print(f"Daten-Ordner: {DATA_DIR}")
print(f"Modelle-Ordner: {MODELS_DIR}")



Projekt-Root: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr
Scripts-Ordner: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\scripts
Daten-Ordner: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\data
Modelle-Ordner: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\models


In [20]:
import random
import pickle
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Projekt-Module
from model import HeteroGNNModel
from dataloader import create_dataloaders, ShiftDataset
from train import train_model, extract_training_config

# Für GNNExplainer
from torch_geometric.explain import Explainer, HeteroExplanation
from torch_geometric.explain.algorithm import GNNExplainer
from torch_geometric.explain.config import ModelConfig, ModelMode, ModelReturnType, ModelTaskLevel

from explainer_utils import (
    NodeTypeRegressionWrapper,
    build_dataset,
    ensure_dir,
    get_device,
    heterodata_to_dicts,
    load_config,
    load_stats,
    load_trained_model,
    summarize_incident_edges,
    summarize_node_mask,
    validate_indices,
)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.4.1
CUDA verfügbar: False


---
## 1. Modell Training

Trainiere ein heterogenes GNN-Modell zur Vorhersage von chemischen Shifts.

### 1.1 Training Parameter

In [21]:
# === WandB Konfiguration ===
use_wandb_widget = widgets.Checkbox(
    value=True,
    description='WandB verwenden',
    style={'description_width': 'initial'}
)

wandb_project_widget = widgets.Text(
    value='ground truth',
    description='WandB Projekt:',
    style={'description_width': 'initial'}
)

# === Grundlegende Parameter ===
seed_widget = widgets.IntText(
    value=0,
    description='Random Seed:',
    style={'description_width': 'initial'}
)

batch_size_widget = widgets.IntSlider(
    value=2,
    min=1,
    max=32,
    description='Batch Size:',
    style={'description_width': 'initial'}
)

num_epochs_widget = widgets.IntSlider(
    value=80,
    min=10,
    max=500,
    step=10,
    description='Epochen:',
    style={'description_width': 'initial'}
)

lr_widget = widgets.FloatLogSlider(
    value=2e-4,
    base=10,
    min=-5,
    max=-2,
    step=0.1,
    description='Learning Rate:',
    style={'description_width': 'initial'},
    readout_format='.2e'
)

# === Modell-Architektur ===
hidden_dim_widget = widgets.Dropdown(
    options=[32, 64, 128, 256, 512],
    value=128,
    description='Hidden Dim:',
    style={'description_width': 'initial'}
)

out_dim_widget = widgets.Dropdown(
    options=[32, 64, 128, 256, 512],
    value=128,
    description='Output Dim:',
    style={'description_width': 'initial'}
)

num_gnn_layers_widget = widgets.IntSlider(
    value=3,
    min=1,
    max=8,
    description='GNN Layers:',
    style={'description_width': 'initial'}
)

operator_type_widget = widgets.Dropdown(
    options=['SAGEConv', 'GCNConv', 'GATConv', 'GATv2Conv', 'GraphConv', 'NNConv', 'GINEConv', 'TransformerConv'],
    value='GraphConv',
    description='Operator:',
    style={'description_width': 'initial'}
)

# === Dropout ===
encoder_dropout_widget = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=0.5,
    step=0.05,
    description='Encoder Dropout:',
    style={'description_width': 'initial'}
)

gnn_dropout_widget = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=0.5,
    step=0.05,
    description='GNN Dropout:',
    style={'description_width': 'initial'}
)

# === Optimizer & Scheduler ===
optimizer_widget = widgets.Dropdown(
    options=['Adam', 'SGD', 'AdamW'],
    value='Adam',
    description='Optimizer:',
    style={'description_width': 'initial'}
)

weight_decay_widget = widgets.FloatLogSlider(
    value=5e-5,
    base=10,
    min=-6,
    max=-2,
    step=0.1,
    description='Weight Decay:',
    style={'description_width': 'initial'},
    readout_format='.2e'
)

scheduler_factor_widget = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=0.9,
    step=0.1,
    description='Scheduler Factor:',
    style={'description_width': 'initial'}
)

scheduler_patience_widget = widgets.IntSlider(
    value=15,
    min=5,
    max=50,
    description='Scheduler Patience:',
    style={'description_width': 'initial'}
)

# === Loss Weights ===
loss_weight_H_widget = widgets.FloatSlider(
    value=10.0,
    min=1.0,
    max=20.0,
    step=1.0,
    description='Loss Weight H:',
    style={'description_width': 'initial'}
)

loss_weight_C_widget = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description='Loss Weight C:',
    style={'description_width': 'initial'}
)

# === Normalisierung ===
normalize_nodes_widget = widgets.Checkbox(
    value=True,
    description='Node Features normalisieren',
    style={'description_width': 'initial'}
)

normalize_edges_widget = widgets.Checkbox(
    value=True,
    description='Edge Features normalisieren',
    style={'description_width': 'initial'}
)

# === Split Ratio ===
train_ratio_widget = widgets.FloatSlider(
    value=0.7,
    min=0.5,
    max=0.95,
    step=0.025,
    description='Train Ratio:',
    style={'description_width': 'initial'}
)

val_ratio_widget = widgets.FloatSlider(
    value=0.15,
    min=0.0,
    max=0.3,
    step=0.025,
    description='Val Ratio:',
    style={'description_width': 'initial'}
)

# === Output ===
output_predictions_widget = widgets.Checkbox(
    value=False,
    description='Detaillierte Vorhersagen speichern',
    style={'description_width': 'initial'}
)

output_dir_widget = widgets.Text(
    value='results',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Layout erstellen
wandb_box = widgets.VBox([
    widgets.HTML('<h4>WandB Konfiguration</h4>'),
    use_wandb_widget,
    wandb_project_widget
])

basic_box = widgets.VBox([
    widgets.HTML('<h4>Grundeinstellungen</h4>'),
    seed_widget,
    batch_size_widget,
    num_epochs_widget,
    lr_widget
])

model_box = widgets.VBox([
    widgets.HTML('<h4>Modell-Architektur</h4>'),
    hidden_dim_widget,
    out_dim_widget,
    num_gnn_layers_widget,
    operator_type_widget,
    encoder_dropout_widget,
    gnn_dropout_widget
])

optim_box = widgets.VBox([
    widgets.HTML('<h4>Optimizer & Scheduler</h4>'),
    optimizer_widget,
    weight_decay_widget,
    scheduler_factor_widget,
    scheduler_patience_widget
])

loss_box = widgets.VBox([
    widgets.HTML('<h4>Loss & Normalisierung</h4>'),
    loss_weight_H_widget,
    loss_weight_C_widget,
    normalize_nodes_widget,
    normalize_edges_widget
])

split_box = widgets.VBox([
    widgets.HTML('<h4>Daten-Split & Output</h4>'),
    train_ratio_widget,
    val_ratio_widget,
    output_predictions_widget,
    output_dir_widget
])

# Alles anzeigen
left_column = widgets.VBox([wandb_box, basic_box, model_box])
right_column = widgets.VBox([optim_box, loss_box, split_box])

display(widgets.HBox([left_column, right_column]))

### 1.2 Training starten

In [22]:
def build_config_from_widgets():
    """Erstellt ein Config-Objekt aus den Widget-Werten."""
    class Config:
        pass
    
    config = Config()
    
    # Grundeinstellungen
    config.seed = seed_widget.value
    config.batch_size = batch_size_widget.value
    config.num_epochs = num_epochs_widget.value
    config.lr = lr_widget.value
    
    # Modell-Architektur
    config.hidden_dim = hidden_dim_widget.value
    config.out_dim = out_dim_widget.value
    config.num_gnn_layers = num_gnn_layers_widget.value
    config.operator_type = operator_type_widget.value
    config.encoder_dropout = encoder_dropout_widget.value
    config.gnnlayer_dropout = gnn_dropout_widget.value
    
    # Operator kwargs
    config.operator_kwargs = {}
    if config.operator_type in ['GATConv', 'GATv2Conv']:
        config.operator_kwargs['add_self_loops'] = False
    
    # Optimizer & Scheduler
    config.optimizer = optimizer_widget.value
    config.weight_decay = weight_decay_widget.value
    config.scheduler_factor = scheduler_factor_widget.value
    config.scheduler_patience = scheduler_patience_widget.value
    
    # Loss
    config.loss_weight_H = loss_weight_H_widget.value
    config.loss_weight_C = loss_weight_C_widget.value
    
    # Normalisierung
    config.normalize_node_features = normalize_nodes_widget.value
    config.normalize_edge_features = normalize_edges_widget.value
    
    # Split
    test_ratio = 1.0 - train_ratio_widget.value - val_ratio_widget.value
    config.split_ratio = (train_ratio_widget.value, val_ratio_widget.value, test_ratio)
    
    # Output
    config.output_detailed_predictions = output_predictions_widget.value
    config.output_dir = output_dir_widget.value
    
    # in_dim kommt zentral aus dem Notebook
    config.in_dim_dict = dict(IN_DIM_DICT)
    
    return config


def run_training():
    """Führt das Training mit den aktuellen Widget-Einstellungen durch."""
    
    # WandB initialisieren (optional)
    if use_wandb_widget.value:
        import wandb
        wandb.init(project=wandb_project_widget.value)
        config = wandb.config
        # Widget-Werte in wandb.config übertragen
        for key, value in vars(build_config_from_widgets()).items():
            setattr(config, key, value)
    else:
        config = build_config_from_widgets()
    
    # Seed setzen
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training auf: {device}")
    
    # Dataloaders erstellen
    print("Lade Daten...")
    train_loader, val_loader, test_loader = create_dataloaders(
        batch_size=config.batch_size,
        split_ratio=config.split_ratio,
        normalize_node_features=config.normalize_node_features,
        normalize_edge_features=config.normalize_edge_features
    )
    print(f"Train: {len(train_loader)} Batches, Val: {len(val_loader)} Batches, Test: {len(test_loader)} Batches")
    
    # Modell erstellen
    model = HeteroGNNModel(
        config.in_dim_dict,
        hidden_dim=config.hidden_dim,
        out_dim=config.out_dim,
        encoder_dropout=config.encoder_dropout,
        gnnlayer_dropout=config.gnnlayer_dropout,
        num_gnn_layers=config.num_gnn_layers,
        operator_type=config.operator_type,
        operator_kwargs=config.operator_kwargs,
        edge_in_dim=10
    ).to(device)
    
    print(f"\nModell erstellt: {config.operator_type} mit {config.num_gnn_layers} Layern")
    
    # Training
    print("\nStarte Training...")
    trained_model = train_model(
        model,
        train_loader,
        val_loader,
        test_loader,
        device,
        config
    )
    
    # Config speichern für spätere Verwendung
    config_path = MODELS_DIR / 'config.pkl'
    os.makedirs(MODELS_DIR, exist_ok=True)
    with open(config_path, 'wb') as f:
        cfg = extract_training_config(config)
        cfg['in_dim_dict'] = dict(config.in_dim_dict)
        pickle.dump(cfg, f)
    print(f"\nKonfiguration gespeichert: {config_path}")
    
    # WandB beenden
    if use_wandb_widget.value:
        wandb.finish()
    
    return trained_model, config


# Training-Button
train_button = widgets.Button(
    description='Training starten',
    button_style='success',
    icon='play'
)

train_output = widgets.Output()

def on_train_click(b):
    with train_output:
        clear_output()
        try:
            global trained_model, training_config
            trained_model, training_config = run_training()
            print("\n✅ Training abgeschlossen!")
        except Exception as e:
            print(f"\n❌ Fehler beim Training: {e}")
            raise

train_button.on_click(on_train_click)

display(train_button)
display(train_output)

Button(button_style='success', description='Training starten', icon='play', style=ButtonStyle())

Output()

---
## 2. Vorhersagen mit trainiertem Modell

Lade ein trainiertes Modell und erstelle Vorhersagen für neue Daten.

### 2.1 Vorhersage Parameter

In [23]:
# Verfügbare Modelle und Daten auflisten
def list_files(directory, extension):
    """Listet Dateien mit bestimmter Endung in einem Verzeichnis."""
    path = Path(directory)
    if path.exists():
        return [f.name for f in path.glob(f'*{extension}')]
    return []

# Widgets für Vorhersage
model_files = list_files(MODELS_DIR, '.pt')
data_files = list_files(DATA_DIR, '.pkl')

pred_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

pred_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

pred_output_widget = widgets.Text(
    value='predictions.csv',
    description='Output Datei:',
    style={'description_width': 'initial'}
)

# Refresh Button
refresh_button = widgets.Button(
    description='Dateien aktualisieren',
    button_style='info',
    icon='refresh'
)

def on_refresh_click(b):
    model_files = list_files(MODELS_DIR, '.pt')
    data_files = list_files(DATA_DIR, '.pkl')
    pred_model_widget.options = model_files if model_files else ['Keine Modelle gefunden']
    pred_data_widget.options = data_files if data_files else ['Keine Daten gefunden']
    # Auch für Explainer aktualisieren
    exp_model_widget.options = model_files if model_files else ['Keine Modelle gefunden']
    exp_data_widget.options = data_files if data_files else ['Keine Daten gefunden']

refresh_button.on_click(on_refresh_click)

display(widgets.VBox([
    widgets.HTML('<h4>Vorhersage Einstellungen</h4>'),
    refresh_button,
    pred_model_widget,
    pred_data_widget,
    pred_output_widget
]))

### 2.2 Vorhersagen erstellen

In [24]:
def run_prediction():
    """Führt Vorhersagen mit dem ausgewählten Modell durch."""
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Pfade
    model_path = MODELS_DIR / pred_model_widget.value
    data_path = DATA_DIR / pred_data_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'
    
    # Überprüfungen
    for path, name in [(model_path, 'Modell'), (data_path, 'Daten'), 
                       (config_path, 'Config'), (norm_stats_path, 'Norm Stats'),
                       (edge_stats_path, 'Edge Stats')]:
        if not path.exists():
            raise FileNotFoundError(f"{name} nicht gefunden: {path}")
    
    # Config laden
    with open(config_path, 'rb') as f:
        config = pickle.load(f)
    # NEU: dict-sicher machen
    if not isinstance(config, dict):
        config = vars(config) if hasattr(config, '__dict__') else dict(config)

    # Stats laden
    with open(norm_stats_path, 'rb') as f:
        norm_stats = pickle.load(f)
    with open(edge_stats_path, 'rb') as f:
        edge_stats = pickle.load(f)
    
    # Dataset erstellen
    dataset = ShiftDataset(
        root_dir=str(DATA_DIR),
        file_name=pred_data_widget.value,
        # NEU: dict.get statt Attribute
        normalize_node_features=config.get('normalize_node_features', True),
        normalize_edge_features=config.get('normalize_edge_features', True),
        norm_stats=norm_stats,
        **edge_stats
    )
    
    # Modell erstellen und laden
    in_dim_dict = config.get('in_dim_dict', IN_DIM_DICT)
    
    operator_kwargs = {}
    operator_type = config.get('operator_type', 'GATv2Conv')  # NEU
    if operator_type in ['GATConv', 'GATv2Conv']:
        operator_kwargs['add_self_loops'] = False
    
    model = HeteroGNNModel(
        in_dim_dict,
        hidden_dim=config.get('hidden_dim', 128),          # NEU
        out_dim=config.get('out_dim', 1),                  # NEU
        encoder_dropout=config.get('encoder_dropout', 0.0),# NEU
        gnnlayer_dropout=config.get('gnnlayer_dropout', 0.0),# NEU
        num_gnn_layers=config.get('num_gnn_layers', 3),    # NEU
        operator_type=operator_type,
        operator_kwargs=operator_kwargs,
        edge_in_dim=10
    )
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    print(f"Modell geladen: {pred_model_widget.value}")
    print(f"Daten: {pred_data_widget.value} ({len(dataset)} Graphen)")
    
    results = []
    
    with torch.no_grad():
        for idx in range(len(dataset)):
            nx_g = dataset.nx_graphs[idx]
            compound = nx_g.graph.get("compound", f"unknown_{idx}")
            structure = nx_g.graph.get("structure", "unknown")
            
            data = dataset[idx].to(device)
            
            x_dict = {ntype: data[ntype].x for ntype in data.node_types}
            edge_index_dict = {}
            edge_attr_dict = {}
            for store in data.edge_stores:
                src, rel, dst = store._key
                edge_index_dict[(src, rel, dst)] = store.edge_index
                edge_attr_dict[(src, rel, dst)] = store.edge_attr
            
            out_dict = model(x_dict, edge_index_dict, edge_attr_dict)
            
            # Knoten sammeln
            h_nodes, c_nodes = [], []
            for node in nx_g.nodes():
                element = nx_g.nodes[node]["element"]
                if element == "H":
                    h_nodes.append(node)
                elif element == "C":
                    c_nodes.append(node)
            
            # H-Vorhersagen
            if 'H' in out_dict and out_dict['H'] is not None:
                for i, node in enumerate(h_nodes):
                    attrs = nx_g.nodes[node]
                    shift_low = attrs.get("shift_low", 0.0)
                    prediction = out_dict['H'][i].item() if i < out_dict['H'].shape[0] else float('nan')
                    results.append({
                        'compound': compound,
                        'structure': structure,
                        'atom_type': 'H',
                        'atom_idx': node,
                        'shift_low': shift_low,
                        'prediction': prediction
                    })
            
            # C-Vorhersagen
            if 'C' in out_dict and out_dict['C'] is not None:
                for i, node in enumerate(c_nodes):
                    attrs = nx_g.nodes[node]
                    shift_low = attrs.get("shift_low", 0.0)
                    prediction = out_dict['C'][i].item() if i < out_dict['C'].shape[0] else float('nan')
                    results.append({
                        'compound': compound,
                        'structure': structure,
                        'atom_type': 'C',
                        'atom_idx': node,
                        'shift_low': shift_low,
                        'prediction': prediction
                    })
    
    df = pd.DataFrame(results)
    output_path = PROJECT_ROOT / pred_output_widget.value
    df.to_csv(output_path, index=False)
    
    print(f"\n✅ {len(results)} Vorhersagen gespeichert: {output_path}")
    
    return df


# Prediction Button
predict_button = widgets.Button(
    description='Vorhersagen erstellen',
    button_style='primary',
    icon='calculator'
)

predict_output = widgets.Output()

def on_predict_click(b):
    with predict_output:
        clear_output()
        try:
            global predictions_df
            predictions_df = run_prediction()
            print("\nVorschau der Ergebnisse:")
            display(predictions_df.head(10))
        except Exception as e:
            print(f"\n❌ Fehler bei Vorhersage: {e}")
            raise

predict_button.on_click(on_predict_click)

display(predict_button)
display(predict_output)

Button(button_style='primary', description='Vorhersagen erstellen', icon='calculator', style=ButtonStyle())

Output()

---
## 3. GNNExplainer - Erklärungen generieren

Erkläre Vorhersagen für einzelne Knoten mittels GNNExplainer.

### 3.1 Explainer Parameter

In [25]:
# Explainer Widgets
exp_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

exp_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

exp_graph_idx_widget = widgets.IntText(
    value=0,
    description='Graph Index:',
    style={'description_width': 'initial'}
)

exp_node_type_widget = widgets.Dropdown(
    options=['H', 'C', 'Others'],
    value='H',
    description='Node Type:',
    style={'description_width': 'initial'}
)

exp_node_idx_widget = widgets.IntText(
    value=0,
    description='Node Index:',
    style={'description_width': 'initial'}
)

exp_epochs_widget = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description='Epochen:',
    style={'description_width': 'initial'}
)

exp_lr_widget = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-3,
    max=-1,
    step=0.1,
    description='Learning Rate:',
    style={'description_width': 'initial'},
    readout_format='.3f'
)

exp_type_widget = widgets.Dropdown(
    options=['phenomenon', 'model'],
    value='phenomenon',
    description='Explanation Type:',
    style={'description_width': 'initial'}
)

exp_topk_features_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    description='Top-K Features:',
    style={'description_width': 'initial'}
)

exp_topk_edges_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    description='Top-K Edges:',
    style={'description_width': 'initial'}
)

exp_output_dir_widget = widgets.Text(
    value='results/explanations',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Layout
exp_left = widgets.VBox([
    widgets.HTML('<h4>Daten & Modell</h4>'),
    exp_model_widget,
    exp_data_widget,
    exp_graph_idx_widget,
    exp_node_type_widget,
    exp_node_idx_widget
])

exp_right = widgets.VBox([
    widgets.HTML('<h4>Explainer Einstellungen</h4>'),
    exp_epochs_widget,
    exp_lr_widget,
    exp_type_widget,
    exp_topk_features_widget,
    exp_topk_edges_widget,
    exp_output_dir_widget
])

display(widgets.HBox([exp_left, exp_right]))


### 3.2 Erklärung generieren

In [26]:
def run_explanation(explain_all_nodes=False):
    '''Generiert eine GNNExplainer Erklärung für den ausgewählten Knoten oder alle Knoten.'''
    
    device = get_device(None)
    
    # Pfade
    model_path = MODELS_DIR / exp_model_widget.value
    data_path = DATA_DIR / exp_data_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'
    
    # Config und Stats laden
    config = load_config(str(config_path))
    norm_stats, edge_stats = load_stats(str(norm_stats_path), str(edge_stats_path))
    
    # Dataset erstellen
    dataset = build_dataset(str(data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
    validate_indices(len(dataset), exp_graph_idx_widget.value, 'graph_idx')
    
    # Daten laden
    data = dataset[exp_graph_idx_widget.value].to(device)
    x_dict, edge_index_dict, edge_attr_dict, y_dict = heterodata_to_dicts(data)
    
    # Modell laden
    base_model = load_trained_model(str(model_path), config, device)
    
    # Wenn explain_all_nodes, sammle alle Erklärungen für alle Node-Typen
    if explain_all_nodes:
        print("Generiere Erklärungen für ALLE Nodes...")
        all_summaries = []
        node_types_to_explain = ['H', 'C', 'Others']
        
        # Sammle alle important_edges aus allen Nodes
        combined_important_edges = []
        
        for node_type in node_types_to_explain:
            if node_type not in x_dict:
                continue
            
            num_nodes = x_dict[node_type].size(0)
            print(f"  Erkläre {num_nodes} Nodes vom Typ {node_type}...")
            
            wrapped_model = NodeTypeRegressionWrapper(base_model, node_type)
            target = None
            target_tensor = y_dict.get(node_type)
            
            if target_tensor is not None and exp_type_widget.value == 'phenomenon':
                target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
                target = target_tensor
            
            model_config = ModelConfig(
                mode=ModelMode.regression,
                task_level=ModelTaskLevel.node,
                return_type=ModelReturnType.raw,
            )
            
            explainer = Explainer(
                model=wrapped_model,
                algorithm=GNNExplainer(epochs=exp_epochs_widget.value, lr=exp_lr_widget.value),
                explanation_type=exp_type_widget.value,
                model_config=model_config,
                node_mask_type='attributes',
                edge_mask_type='object',
            )
            
            for node_idx in range(num_nodes):
                try:
                    if target is not None and torch.isnan(target[node_idx]):
                        continue
                    
                    explanation = explainer(
                        x_dict,
                        edge_index_dict,
                        edge_attr_dict=edge_attr_dict,
                        target=target,
                        index=node_idx,
                    )
                    
                    # Edge summary für diesen Node
                    edge_summary = []
                    for edge_type, mask in explanation.edge_mask_dict.items():
                        if mask is None:
                            continue
                        edge_index = edge_index_dict.get(edge_type)
                        if edge_index is None:
                            continue
                        mask_vals = mask.view(-1).detach().cpu()
                        rows = edge_index[0].detach().cpu()
                        cols = edge_index[1].detach().cpu()
                        for edge_pos, importance in enumerate(mask_vals):
                            edge_summary.append({
                                'edge_type': edge_type,
                                'edge_position': edge_pos,
                                'importance': float(importance),
                                'src_index': int(rows[edge_pos]),
                                'dst_index': int(cols[edge_pos]),
                            })
                    
                    edge_summary.sort(key=lambda item: item['importance'], reverse=True)
                    
                    # Füge alle Edges zur kombinierten Liste hinzu
                    combined_important_edges.extend(edge_summary)
                    
                    summary = {
                        'graph_idx': exp_graph_idx_widget.value,
                        'node_type': node_type,
                        'node_idx': node_idx,
                        'important_edges': edge_summary,
                    }
                    
                    all_summaries.append(summary)
                    
                except Exception as e:
                    print(f"  Fehler bei {node_type}[{node_idx}]: {e}")
                    continue
        
        print(f"✅ {len(all_summaries)} Erklärungen generiert")
        
        # Erstelle eine kombinierte Summary mit allen Edges
        combined_summary = {
            'graph_idx': exp_graph_idx_widget.value,
            'node_type': 'ALL',
            'node_idx': -1,
            'important_edges': combined_important_edges,
        }
        
        # Speichere global
        global all_node_summaries
        all_node_summaries = all_summaries
        
        # Speichere auch ein dummy explanation object
        global explanation_obj
        if all_summaries:
            # Nutze die erste Node-Explanation als Basis
            wrapped_model = NodeTypeRegressionWrapper(base_model, all_summaries[0]['node_type'])
            target_tensor = y_dict.get(all_summaries[0]['node_type'])
            target = None
            if target_tensor is not None and exp_type_widget.value == 'phenomenon':
                target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
                target = target_tensor
            
            model_config = ModelConfig(
                mode=ModelMode.regression,
                task_level=ModelTaskLevel.node,
                return_type=ModelReturnType.raw,
            )
            
            explainer = Explainer(
                model=wrapped_model,
                algorithm=GNNExplainer(epochs=exp_epochs_widget.value, lr=exp_lr_widget.value),
                explanation_type=exp_type_widget.value,
                model_config=model_config,
                node_mask_type='attributes',
                edge_mask_type='object',
            )
            
            explanation_obj = explainer(
                x_dict,
                edge_index_dict,
                edge_attr_dict=edge_attr_dict,
                target=target,
                index=all_summaries[0]['node_idx'],
            )
        
        return combined_summary, explanation_obj
    
    # Einzelner Node (originales Verhalten)
    wrapped_model = NodeTypeRegressionWrapper(base_model, exp_node_type_widget.value)
    
    # Target vorbereiten
    target = None
    target_tensor = y_dict.get(exp_node_type_widget.value)
    
    if target_tensor is not None:
        target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
        validate_indices(target_tensor.size(0), exp_node_idx_widget.value, 'node_idx')
        if exp_type_widget.value == 'phenomenon':
            if torch.isnan(target_tensor[exp_node_idx_widget.value]):
                raise ValueError(
                    f'Target für Knoten {exp_node_idx_widget.value} ist NaN. '
                    "Wähle einen anderen Knoten oder nutze 'model' als Explanation Type."
                )
            target = target_tensor
    else:
        validate_indices(x_dict[exp_node_type_widget.value].size(0), exp_node_idx_widget.value, 'node_idx')
        if exp_type_widget.value == 'phenomenon':
            raise ValueError(
                f'Keine Targets für Node Type {exp_node_type_widget.value}; '
                'kann phenomenon nicht erklären.'
            )
    
    # Explainer konfigurieren
    model_config = ModelConfig(
        mode=ModelMode.regression,
        task_level=ModelTaskLevel.node,
        return_type=ModelReturnType.raw,
    )
    
    explainer = Explainer(
        model=wrapped_model,
        algorithm=GNNExplainer(epochs=exp_epochs_widget.value, lr=exp_lr_widget.value),
        explanation_type=exp_type_widget.value,
        model_config=model_config,
        node_mask_type='attributes',
        edge_mask_type='object',
    )
    
    print(f'Generiere Erklärung für {exp_node_type_widget.value}[{exp_node_idx_widget.value}]...')
    
    # Erklärung generieren
    explanation = explainer(
        x_dict,
        edge_index_dict,
        edge_attr_dict=edge_attr_dict,
        target=target,
        index=exp_node_idx_widget.value,
    )
    
    if not isinstance(explanation, HeteroExplanation):
        raise TypeError(
            f'Erwartete HeteroExplanation, erhielt {type(explanation)}. '
            'Stelle sicher, dass eine aktuelle torch_geometric Version verwendet wird.'
        )
    
    # Vorhersage holen
    with torch.no_grad():
        predictions = base_model(x_dict, edge_index_dict, edge_attr_dict)
        node_prediction = float(predictions[exp_node_type_widget.value][exp_node_idx_widget.value].item())
    
    # Zusammenfassungen erstellen
    feature_summary = summarize_node_mask(
        explanation.node_mask_dict,
        exp_node_type_widget.value,
        exp_node_idx_widget.value,
        top_k=max(exp_topk_features_widget.value, 0),
    )
    
    # Alle Kanten aus der Edge-Mask sammeln (keine Filter), danach nach Importance sortieren
    edge_summary = []
    for edge_type, mask in explanation.edge_mask_dict.items():
        if mask is None:
            continue
        edge_index = edge_index_dict.get(edge_type)
        if edge_index is None:
            continue
        mask_vals = mask.view(-1).detach().cpu()
        rows = edge_index[0].detach().cpu()
        cols = edge_index[1].detach().cpu()
        for edge_pos, importance in enumerate(mask_vals):
            edge_summary.append(
                {
                    'edge_type': edge_type,
                    'edge_position': edge_pos,
                    'importance': float(importance),
                    'src_index': int(rows[edge_pos]),
                    'dst_index': int(cols[edge_pos]),
                }
            )
    edge_summary.sort(key=lambda item: item['importance'], reverse=True)
    
    # ASCII-Tabelle zur schnellen Sichtkontrolle (gekürzt auf 40 Zeilen)
    edge_table_rows: List[Dict[str, Any]] = []
    for entry in edge_summary:
        src_type, _rel_type, dst_type = entry['edge_type']
        src_idx = entry.get('src_index')
        dst_idx = entry.get('dst_index')
        edge_table_rows.append(
            {
                'src_label': f'{src_type}[{src_idx}]',
                'dst_label': f'{dst_type}[{dst_idx}]',
                'edge_pos': entry.get('edge_position'),
                'importance': entry.get('importance'),
            }
        )
    max_rows_print = 40
    col_specs = [
        ('src_label', 14),
        ('dst_label', 14),
        ('edge_pos', 9),
        ('importance', 12),
    ]
    def fmt_row(row_dict):
        cells = []
        for key, width in col_specs:
            val = row_dict[key]
            if key == 'importance':
                cells.append(f"{val:>{width}.6f}")
            else:
                cells.append(f"{str(val):<{width}}")
        return ' '.join(cells)
    header = ' '.join(f"{name:<{width}}" for name, width in col_specs)
    divider = '-' * len(header)
    table_lines = ['EDGE TABLE (top nach Importance, gekürzt)', header, divider]
    for row in edge_table_rows[:max_rows_print]:
        table_lines.append(fmt_row(row))
    if len(edge_table_rows) > max_rows_print:
        table_lines.append(f"... ({len(edge_table_rows) - max_rows_print} weitere Zeilen)")
    print('\n'.join(table_lines))
    
    target_value = float(target[exp_node_idx_widget.value].item()) if target is not None else None
    
    summary = {
        'graph_idx': exp_graph_idx_widget.value,
        'node_type': exp_node_type_widget.value,
        'node_idx': exp_node_idx_widget.value,
        'prediction': node_prediction,
        'target': target_value,
        'top_features': feature_summary,
        'important_edges': edge_summary,
    }
    
    # Speichern (.pt + JSON mit Summary/Edges)
    output_dir = PROJECT_ROOT / exp_output_dir_widget.value
    ensure_dir(str(output_dir))
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base_name = f'gnn_explainer_{exp_node_type_widget.value}_n{exp_node_idx_widget.value}_g{exp_graph_idx_widget.value}_{timestamp}'
    
    torch.save(explanation, output_dir / f'{base_name}.pt')
    with open(output_dir / f'{base_name}.json', 'w', encoding='utf-8') as handle:
        json.dump(summary, handle, indent=2)
    
    print(f"\n✅ Erklärung gespeichert in: {output_dir}")
    
    return summary, explanation

# Explain Button
explain_button = widgets.Button(
    description='Erklärung generieren',
    button_style='warning',
    icon='lightbulb'
    )

explain_output = widgets.Output()

def on_explain_click(b):
    with explain_output:
        clear_output()
        try:
            global explanation_summary, explanation_obj
            explanation_summary, explanation_obj = run_explanation()
            print('\n' + '='*50)
            print('ERKLÄRUNGSZUSAMMENFASSUNG')
            print('='*50)
            print(json.dumps(explanation_summary, indent=2))
        except Exception as e:
            print(f'\n❌ Fehler bei Erklärung: {e}')
            raise

explain_button.on_click(on_explain_click)

display(explain_button)
display(explain_output)


Button(button_style='warning', description='Erklärung generieren', icon='lightbulb', style=ButtonStyle())

Output()

### Visualisierung

### Batch Analysis

In [27]:
# Batch analysis widgets
batch_node_types_widget = widgets.SelectMultiple(
    options=['H', 'C', 'Others'],
    value=['H', 'C'],
    description='Node types:',
    style={'description_width': 'initial'}
)

batch_epochs_widget = widgets.IntSlider(
    value=50,
    min=10,
    max=100,
    step=10,
    description='Epochs per explanation:',
    style={'description_width': 'initial'}
)

batch_sample_graphs_widget = widgets.IntText(
    value=0,
    description='Sample graphs (0=all):',
    style={'description_width': 'initial'}
)

batch_sample_nodes_widget = widgets.IntText(
    value=0,
    description='Max nodes per graph (0=all):',
    style={'description_width': 'initial'}
)

batch_target_nodes_widget = widgets.Textarea(
    value='',
    description='Target nodes:',
    placeholder='Examples: 3:0,5; 7:H:2',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', height='70px')
)

batch_target_nodes_help = widgets.HTML(
    "Format: graph_idx:node_idx,node_idx; graph_idx:node_type:node_idx"
    " (node_type optional; empty uses sampling)"
)

batch_type_widget = widgets.Dropdown(
    options=['phenomenon', 'model'],
    value='phenomenon',
    description='Explanation Type:',
    style={'description_width': 'initial'}
)

display(widgets.VBox([
    widgets.HTML('<h4>Batch Analysis Configuration</h4>'),
    batch_node_types_widget,
    batch_epochs_widget,
    batch_sample_graphs_widget,
    batch_sample_nodes_widget,
    batch_target_nodes_widget,
    batch_target_nodes_help,
    batch_type_widget
]))



In [28]:
from feature_visualization import get_feature_names, plot_feature_importance

def parse_target_nodes(text, allowed_node_types):
    '''Parse target nodes into a mapping of graph -> node types -> nodes.'''
    targets = {}
    explicit_node_types = []
    raw = (text or '').strip()
    if not raw:
        return targets, explicit_node_types

    entries = [entry.strip() for entry in raw.replace("\n", ";").split(";") if entry.strip()]
    for entry in entries:
        parts = [part.strip() for part in entry.split(':') if part.strip()]
        if len(parts) == 2:
            graph_part, nodes_part = parts
            node_type = None
        elif len(parts) == 3:
            graph_part, node_type, nodes_part = parts
            if node_type not in allowed_node_types:
                raise ValueError(f"Unknown node type '{node_type}' in target nodes.")
            if node_type not in explicit_node_types:
                explicit_node_types.append(node_type)
        else:
            raise ValueError(
                "Invalid target format. Use graph_idx:node_idx,node_idx or "
                "graph_idx:node_type:node_idx."
            )

        graph_idx = int(graph_part)
        node_indices = []
        for token in nodes_part.replace(',', ' ').split():
            node_indices.append(int(token))
        if not node_indices:
            raise ValueError(f"No node indices provided in entry '{entry}'.")

        targets.setdefault(graph_idx, {})
        targets[graph_idx].setdefault(node_type, set()).update(node_indices)

    return targets, explicit_node_types


def run_batch_analysis():
    '''Run feature importance analysis across all selected nodes.'''

    from tqdm.notebook import tqdm

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Load model and data
    model_path = MODELS_DIR / exp_model_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'
    data_path = DATA_DIR / exp_data_widget.value

    config = load_config(str(config_path))
    norm_stats, edge_stats = load_stats(str(norm_stats_path), str(edge_stats_path))
    dataset = build_dataset(str(data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)

    base_model = load_trained_model(str(model_path), config, device)

    # Configuration
    allowed_node_types = list(batch_node_types_widget.options)
    node_types = list(batch_node_types_widget.value)
    epochs = batch_epochs_widget.value
    sample_graphs = batch_sample_graphs_widget.value
    max_nodes_per_graph = batch_sample_nodes_widget.value
    expl_type = batch_type_widget.value

    target_nodes, explicit_node_types = parse_target_nodes(
        batch_target_nodes_widget.value, allowed_node_types
    )
    for node_type in explicit_node_types:
        if node_type not in node_types:
            print(f"Adding node type '{node_type}' from target nodes.")
            node_types.append(node_type)

    # Aggregate importance scores per node type
    aggregate_importance = {nt: [] for nt in node_types}

    # Sample graphs if requested (ignored when target nodes are provided)
    if target_nodes:
        graphs_to_process = sorted(target_nodes.keys())
    else:
        graphs_to_process = list(range(len(dataset)))
        if sample_graphs > 0 and sample_graphs < len(graphs_to_process):
            graphs_to_process = graphs_to_process[:sample_graphs]

    total_explanations = 0

    progress = tqdm(total=len(graphs_to_process), desc='Processing graphs')

    for graph_idx in graphs_to_process:
        validate_indices(len(dataset), graph_idx, "graph_idx")
        data = dataset[graph_idx].to(device)
        x_dict, edge_index_dict, edge_attr_dict, y_dict = heterodata_to_dicts(data)

        graph_targets = target_nodes.get(graph_idx, {}) if target_nodes else {}

        for node_type in node_types:
            if node_type not in x_dict:
                continue
            wrapped_model = NodeTypeRegressionWrapper(base_model, node_type)

            target = None
            target_tensor = y_dict.get(node_type)
            if target_tensor is not None and expl_type == 'phenomenon':
                target = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)

            num_nodes = x_dict[node_type].size(0)
            if graph_targets:
                nodes_to_process = set(graph_targets.get(None, set()))
                if node_type in graph_targets:
                    nodes_to_process.update(graph_targets[node_type])
                if not nodes_to_process:
                    continue
                nodes_to_process = sorted(nodes_to_process)
            else:
                nodes_to_process = list(range(num_nodes))
                if max_nodes_per_graph > 0 and max_nodes_per_graph < num_nodes:
                    import random
                    random.seed(42)
                    nodes_to_process = random.sample(nodes_to_process, max_nodes_per_graph)

            for node_idx in nodes_to_process:
                if node_idx < 0 or node_idx >= num_nodes:
                    print(
                        f"Skipping graph {graph_idx}, node_type {node_type}, node {node_idx}: "
                        "out of bounds"
                    )
                    continue
                # Skip invalid targets
                if target is not None and torch.isnan(target[node_idx]):
                    continue

                try:
                    # Configure explainer
                    model_config = ModelConfig(
                        mode=ModelMode.regression,
                        task_level=ModelTaskLevel.node,
                        return_type=ModelReturnType.raw,
                    )

                    explainer = Explainer(
                        model=wrapped_model,
                        algorithm=GNNExplainer(epochs=epochs, lr=0.01),
                        explanation_type=expl_type,
                        model_config=model_config,
                        node_mask_type='attributes',
                        edge_mask_type='object',
                    )

                    explanation = explainer(
                        x_dict,
                        edge_index_dict,
                        edge_attr_dict=edge_attr_dict,
                        target=target,
                        index=node_idx,
                    )

                    # Extract feature importance
                    mask = explanation.node_mask_dict.get(node_type)
                    if mask is not None and mask.dim() == 2:
                        importance = mask[node_idx].cpu().detach().numpy()
                    elif mask is not None:
                        importance = mask.cpu().detach().numpy()
                    else:
                        continue

                    aggregate_importance[node_type].append(importance)
                    total_explanations += 1

                except Exception as e:
                    print(f"Skipped graph {graph_idx}, node_type {node_type}, node {node_idx}: {e}")
                    continue

        progress.update(1)

    progress.close()

    print(f"Completed {total_explanations} explanations")

    # Compute average importance per feature per node type
    results = {}

    for node_type in node_types:
        importances = aggregate_importance[node_type]
        if not importances:
            print(f"No valid explanations for node type {node_type}")
            continue

        import numpy as np
        avg_importance = np.mean(importances, axis=0)
        feature_indices = np.arange(len(avg_importance))

        # Sort by importance (descending for most important, ascending for least)
        most_important = sorted(zip(feature_indices, avg_importance), key=lambda x: x[1], reverse=True)
        least_important = sorted(zip(feature_indices, avg_importance), key=lambda x: x[1])

        results[node_type] = {
            'avg_importance': avg_importance.tolist(),
            'most_important': [{'feature': int(idx), 'importance': float(imp)} for idx, imp in most_important[:10]],
            'least_important': [{'feature': int(idx), 'importance': float(imp)} for idx, imp in least_important[:10]],
                    'explainer_type': 'gnn'
        }

    return results


# Button
batch_button = widgets.Button(
    description='Start Batch Analysis',
    button_style='primary',
    icon='bar-chart'
)

batch_output = widgets.Output()

def on_batch_click(b):
    with batch_output:
        clear_output()
        try:
            global batch_results
            batch_results = run_batch_analysis()

            # Display results
            for node_type, result in batch_results.items():
                print(f"=== Node Type: {node_type} ===")
                print("Top 10 Most Important Features:")
                for item in result['most_important']:
                    print(f"  Feature {item['feature']}: {item['importance']:.4f}")
                print("#Top 10 Least Important Features:")
                for item in result['least_important']:
                    print(f"  Feature {item['feature']}: {item['importance']:.4f}")

            # Balkendiagramme erstellen
            try:
                plot_feature_importance(batch_results, node_type)
            except Exception as e:
                print(f"Fehler bei Visualisierung: {e}")

        except Exception as e:
            print(f"❌ Error in batch analysis: {e}")
            import traceback
            traceback.print_exc()

batch_button.on_click(on_batch_click)

display(batch_button)
display(batch_output)



Button(button_style='primary', description='Start Batch Analysis', icon='bar-chart', style=ButtonStyle())

Output()

In [29]:
def load_sdf_atoms(path):
 
    atoms = []
    with open(path) as f:
        lines = f.readlines()

    if len(lines) < 5:
        return atoms

    # Counts-Zeile (typischerweise Zeile 4, Index 3)
    counts_line = lines[3]
    try:
        n_atoms = int(counts_line[0:3])
    except ValueError:
        # Falls das Format anders ist -> nichts zurückgeben
        return atoms

    # Atomzeilen beginnen ab Zeile 5 (Index 4)
    atom_lines = lines[4 : 4 + n_atoms]

    for idx, line in enumerate(atom_lines):
        # Standard SDF-V2000: x,y,z in Spalten 0:10,10:20,20:30
        try:
            x = float(line[0:10])
            y = float(line[10:20])
            z = float(line[20:30])
            symbol = line[31:34].strip()
        except ValueError:
            continue
        atoms.append(
            {
                "index": idx,
                "symbol": symbol,
                "x": x,
                "y": y,
                "z": z,
            }
        )
    return atoms


# Pure blue→red gradient without green for this cell
# v=0 → blue, v=1 → red
def importance_to_color(v):
    v = max(0.0, min(1.0, float(v)))
    r = int(v * 255)
    g = 0
    b = int((1.0 - v) * 255)
    return f'rgb({r},{g},{b})'


if "explanation_summary" in globals() and "explanation_obj" in globals():
    import py3Dmol
    import pickle
    from pathlib import Path

    PROJECT_ROOT = Path.cwd().parent
    DATA_DIR = PROJECT_ROOT / "data"
    MODELS_DIR = PROJECT_ROOT / "models"

    graph_idx = explanation_summary["graph_idx"]
    node_type = explanation_summary["node_type"]
    node_idx = explanation_summary["node_idx"]

    # Dataset laden
    data_file = exp_data_widget.value
    config_path = MODELS_DIR / "config.pkl"
    norm_stats_path = MODELS_DIR / "norm_stats.pkl"
    edge_stats_path = MODELS_DIR / "edge_stats.pkl"

    with open(config_path, "rb") as f:
        config = pickle.load(f)
    with open(norm_stats_path, "rb") as f:
        norm_stats = pickle.load(f)
    with open(edge_stats_path, "rb") as f:
        edge_stats = pickle.load(f)

    dataset = build_dataset(str(DATA_DIR / data_file), config, norm_stats=norm_stats, edge_stats=edge_stats)
    nx_g = dataset.nx_graphs[graph_idx]

    compound = nx_g.graph.get("compound", 1)
    structure = nx_g.graph.get("structure", 1)
    subfolder = f"{int(compound):03d}"

    sdf_name = f"{subfolder}_{int(structure):02d}.sdf"
    sdf_path = DATA_DIR / "orca_xyz_formate" / subfolder / sdf_name
    print(f"Looking for SDF file: {sdf_path}")

    if sdf_path.exists():
        atoms = load_sdf_atoms(str(sdf_path))

        with open(str(sdf_path)) as f:
            sdf_block = f.read()

        # ERSTELLE ATOM_INDEX_DICT LOKAL aus dem NetworkX-Graphen
        atom_index_dict = {"H": [], "C": [], "Others": []}
        for node in nx_g.nodes():
            element = nx_g.nodes[node]["element"]
            if element in ["H", "C"]:
                atom_index_dict[element].append(node)
            else:
                atom_index_dict["Others"].append(node)
        
        # Konvertiere zu Tensoren
        for ntype in atom_index_dict:
            if atom_index_dict[ntype]:
                atom_index_dict[ntype] = torch.tensor(atom_index_dict[ntype], dtype=torch.long)
            else:
                atom_index_dict[ntype] = torch.empty(0, dtype=torch.long)

        # Fokussiertes Atom
        if node_type in atom_index_dict and atom_index_dict[node_type].numel() > node_idx:
            focus_index = int(atom_index_dict[node_type][node_idx].item())
        else:
            print(f"Fehler: node_idx {node_idx} out of bounds für {node_type}")
            focus_index = None

        # ERSTELLE EDGE_INDEX_DICT LOKAL aus dem HeteroData Objekt
        # Dafür brauchen wir das ursprüngliche data Objekt
        data = dataset[graph_idx]  # HeteroData Objekt
        edge_index_dict = {}
        for store in data.edge_stores:
            key = store._key
            edge_index_dict[key] = store.edge_index

        # Edge-Importances pro Atom aggregieren (Maximum)
        edge_masks = explanation_obj.edge_mask_dict if hasattr(explanation_obj, 'edge_mask_dict') else {}
        atom_importance = {}

        # Fuer jeden Node-Type
        for ntype, atom_indices in atom_index_dict.items():
            if atom_indices.numel() == 0:
                continue
            
            # Edge-Types, die diesen Node-Type betreffen
            relevant_edge_types = []
            for edge_type in edge_index_dict.keys():
                src_type, _, dst_type = edge_type
                if src_type == ntype or dst_type == ntype:
                    relevant_edge_types.append(edge_type)
            
            # Fuer jeden Node des Types
            for local_idx in range(len(atom_indices)):
                sdf_idx = int(atom_indices[local_idx].item())
                
                max_importance = 0.0
                
                # Alle relevanten Edge-Types durchgehen
                for edge_type in relevant_edge_types:
                    if edge_type not in edge_masks:
                        continue
                        
                    ei = edge_index_dict[edge_type]
                    mask = edge_masks[edge_type]
                    
                    # Kanten finden, die diesen Node betreffen
                    src_type, _, dst_type = edge_type
                    
                    if src_type == ntype:
                        # Outgoing edges von diesem Node
                        for edge_idx in range(ei.size(1)):
                            if ei[0, edge_idx] == local_idx:
                                max_importance = max(max_importance, abs(float(mask[edge_idx].item())))
                                
                    if dst_type == ntype:
                        # Incoming edges zu diesem Node
                        for edge_idx in range(ei.size(1)):
                            if ei[1, edge_idx] == local_idx:
                                max_importance = max(max_importance, abs(float(mask[edge_idx].item())))
                
                # Maximaler Importance-Wert der anliegenden Kanten
                atom_importance[sdf_idx] = max_importance

        print(f"Colored atoms (count): {len(atom_importance)}")

        # py3Dmol Viewer Setup
        view = py3Dmol.view(width=600, height=500)
        view.addModel(sdf_block, "sdf")
        view.setStyle({
            "stick": {
                "color": "lightgray",
                "radius": 0.2,
                "doubleBondScaling": 0.4,
                "singleBonds": False,
            }
        })

        # Labels für H + Heteroatome
        if atoms:
            for atom in atoms:
                if atom["symbol"] != "C":
                    view.addLabel(
                        atom["symbol"],
                        {
                            "position": {
                                "x": atom["x"],
                                "y": atom["y"],
                                "z": atom["z"],
                            },
                            "fontSize": 12,
                            "fontColor": "black",
                            "backgroundColor": "white",
                            "showBackground": True,
                        },
                    )

        # Atome einfärben nach Importance
        if atom_importance:
            vals = list(atom_importance.values())
            norm_atoms = make_normalizer(vals)
            for atom_idx, imp in atom_importance.items():
                v = norm_atoms(imp)
                color = importance_to_color(v)
                view.addStyle(
                    {"index": atom_idx},
                    {"stick": {"color": color, "radius": 0.25}},
                )
                view.addStyle(
                    {"index": atom_idx},
                    {"sphere": {"color": color, "radius": 0.4}},
                )

        # Wichtige Kanten zeichnen
        important_edges = []
        if "important_edges" in explanation_summary:
            for e in explanation_summary["important_edges"]:
                important_edges.append({
                    "edge_type": e["edge_type"],
                    "importance": e["importance"],
                    "edge_position": e.get("edge_position", 0),
                })

        if important_edges and atoms:
            imps = [abs(e["importance"]) for e in important_edges]
            norm_edges = make_normalizer(imps)
            
            for e in important_edges:
                edge_type = tuple(e["edge_type"])
                pos = int(e["edge_position"])
                imp = float(e["importance"])
                
                if abs(imp) <= 0.01:
                    continue
                    
                if edge_type not in edge_index_dict:
                    continue

                ei = edge_index_dict[edge_type]
                if pos < 0 or pos >= ei.size(1):
                    continue

                u_local = int(ei[0, pos].item())
                v_local = int(ei[1, pos].item())

                src_type, _, dst_type = edge_type

                # Mapping lokal->SDF ueber das lokale atom_index_dict
                if src_type not in atom_index_dict or atom_index_dict[src_type].numel() <= u_local:
                    continue
                if dst_type not in atom_index_dict or atom_index_dict[dst_type].numel() <= v_local:
                    continue

                u_sdf = int(atom_index_dict[src_type][u_local].item())
                v_sdf = int(atom_index_dict[dst_type][v_local].item())

                if not (0 <= u_sdf < len(atoms) and 0 <= v_sdf < len(atoms)):
                    continue

                v = norm_edges(abs(imp))
                color = importance_to_color(v)

                ai = atoms[u_sdf]
                aj = atoms[v_sdf]
                mid = {
                    "x": (ai["x"] + aj["x"]) / 2,
                    "y": (ai["y"] + aj["y"]) / 2,
                    "z": (ai["z"] + aj["z"]) / 2,
                }

                view.addLine({
                    "start": {"x": ai["x"], "y": ai["y"], "z": ai["z"]},
                    "end": mid,
                    "color": color,
                    "linewidth": 8,
                })

        # Fokus-Atom hervorheben
        if focus_index is not None and 0 <= focus_index < len(atoms):
            fa = atoms[focus_index]
            view.addStyle(
                {"index": focus_index},
                {"sphere": {"color": "yellow", "radius": 0.6}},
            )
            view.addLabel(
                str(focus_index),
                {
                    "fontSize": 14,
                    "fontColor": "black",
                    "backgroundColor": "white",
                    "showBackground": True,
                    "position": {
                        "x": fa["x"],
                        "y": fa["y"],
                        "z": fa["z"],
                    },
                },
            )

        view.zoomTo()
        view.show()
    else:
        print(f"SDF-Datei für Graph {graph_idx} nicht gefunden: {sdf_path}")
else:
    print("Keine Erklärung vorhanden oder Visualisierung mit Beispiel.")

Keine Erklärung vorhanden oder Visualisierung mit Beispiel.


---
## 4. Hilfsfunktionen

In [30]:
def make_normalizer(values, default=1.0):
    if not values:
        return lambda v: default
    vmin = min(values)
    vmax = max(values)

    def norm(v):
        if vmax == vmin:
            return default
        return (v - vmin) / (vmax - vmin)

    return norm


def importance_to_color(v):
    """
    Mappt einen Wert in [0,1] auf ein rot-blau-Farbspektrum:
      v=0 -> blau, v=1 -> rot.
    """
    v = max(0.0, min(1.0, v))
    r = int(v * 255)
    b = int((1 - v) * 255)
    g = 0
    return f"rgb({r},{g},{b})"



def show_dataset_info(data_file):
    """Zeigt Informationen über einen Datensatz an."""
    data_path = DATA_DIR / data_file
    
    if not data_path.exists():
        print(f"Datei nicht gefunden: {data_path}")
        return
    
    # Config laden falls vorhanden
    config_path = MODELS_DIR / 'config.pkl'
    if config_path.exists():
        with open(config_path, 'rb') as f:
            config = pickle.load(f)
        if not isinstance(config, dict):
            config = vars(config) if hasattr(config, '__dict__') else dict(config)
        normalize_nodes = config.get('normalize_node_features', False)
        normalize_edges = config.get('normalize_edge_features', False)
    else:
        normalize_nodes = False
        normalize_edges = False
    
    # Dataset laden
    dataset = ShiftDataset(
        root_dir=str(DATA_DIR),
        file_name=data_file,
        normalize_node_features=normalize_nodes,
        normalize_edge_features=normalize_edges
    )
    
    print(f"Datensatz: {data_file}")
    print(f"Anzahl Graphen: {len(dataset)}")
    
    if len(dataset) > 0:
        sample = dataset[0]
        print(f"\nBeispiel Graph (Index 0):")
        print(f"  Node Types: {sample.node_types}")
        for ntype in sample.node_types:
            print(f"    {ntype}: {sample[ntype].x.shape[0]} Knoten, {sample[ntype].x.shape[1]} Features")
        print(f"  Edge Types: {len(sample.edge_types)}")


# Widget für Dataset Info
info_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Datensatz:',
    style={'description_width': 'initial'}
)

info_button = widgets.Button(
    description='Info anzeigen',
    button_style='',
    icon='info'
)

info_output = widgets.Output()

def on_info_click(b):
    with info_output:
        clear_output()
        show_dataset_info(info_data_widget.value)

info_button.on_click(on_info_click)

display(widgets.VBox([
    widgets.HTML('<h4>Datensatz-Informationen</h4>'),
    info_data_widget,
    info_button,
    info_output
]))

## Integrated Gradients Explainer

In [31]:
# IG Explainer Widgets
ig_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

ig_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

ig_graph_idx_widget = widgets.IntText(
    value=0,
    description='Graph Index:',
    style={'description_width': 'initial'}
)

ig_node_type_widget = widgets.Dropdown(
    options=['H', 'C', 'Others'],
    value='H',
    description='Node Type:',
    style={'description_width': 'initial'}
)

ig_node_idx_widget = widgets.IntText(
    value=0,
    description='Node Index:',
    style={'description_width': 'initial'}
)

ig_n_steps_widget = widgets.IntSlider(
    value=50,
    min=10,
    max=200,
    step=10,
    description='Integration Steps:',
    style={'description_width': 'initial'}
)

ig_output_dir_widget = widgets.Text(
    value='results/explanations',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Layout
ig_left = widgets.VBox([
    widgets.HTML('<h4>Daten & Modell</h4>'),
    ig_model_widget,
    ig_data_widget,
    ig_graph_idx_widget,
    ig_node_type_widget,
    ig_node_idx_widget
])

ig_right = widgets.VBox([
    widgets.HTML('<h4>IG Einstellungen</h4>'),
    ig_n_steps_widget,
    ig_output_dir_widget
])

display(widgets.HBox([ig_left, ig_right]))


In [32]:
from scripts.explainer.ig_explainer import compute_ig_explanation
from feature_visualization import get_feature_names, plot_single_feature_importance

def run_ig_explanation():
    """Generiert eine IG Erklärung für den ausgewählten Knoten."""

    device = get_device(None)

    # Pfade
    model_path = MODELS_DIR / ig_model_widget.value
    data_path = DATA_DIR / ig_data_widget.value
    config_path = MODELS_DIR / 'config.pkl'
    norm_stats_path = MODELS_DIR / 'norm_stats.pkl'
    edge_stats_path = MODELS_DIR / 'edge_stats.pkl'

    # Config und Stats laden
    config = load_config(str(config_path))
    norm_stats, edge_stats = load_stats(str(norm_stats_path), str(edge_stats_path))

    # Dataset erstellen
    dataset = build_dataset(str(data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
    validate_indices(len(dataset), ig_graph_idx_widget.value, "graph_idx")

    # Daten laden
    data = dataset[ig_graph_idx_widget.value].to(device)
    x_dict, edge_index_dict, edge_attr_dict, y_dict = heterodata_to_dicts(data)

    # Modell laden
    base_model = load_trained_model(str(model_path), config, device)

    # IG-Erklärung berechnen
    explanation_result = compute_ig_explanation(
        base_model=base_model,
        data=data,
        node_type=ig_node_type_widget.value,
        node_idx=ig_node_idx_widget.value,
        x_dict=x_dict,
        edge_index_dict=edge_index_dict,
        edge_attr_dict=edge_attr_dict,
        device=device,
        n_steps=ig_n_steps_widget.value
    )

    # Vorhersage holen
    with torch.no_grad():
        predictions = base_model(x_dict, edge_index_dict, edge_attr_dict)
        node_prediction = float(predictions[ig_node_type_widget.value][ig_node_idx_widget.value].item())

    # Ergebnis formatieren
    summary = {
        "graph_idx": ig_graph_idx_widget.value,
        "node_type": ig_node_type_widget.value,
        "node_idx": ig_node_idx_widget.value,
        "prediction": node_prediction,
        "n_steps": ig_n_steps_widget.value,
        "feature_importance": explanation_result["node_mask_dict"][ig_node_type_widget.value][0].tolist(),
    }

    # Speichern
    output_dir = PROJECT_ROOT / ig_output_dir_widget.value
    ensure_dir(str(output_dir))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = (
        f"ig_explainer_{ig_node_type_widget.value}"
        f"_n{ig_node_idx_widget.value}"
        f"_g{ig_graph_idx_widget.value}"
        f"_{timestamp}"
    )

    torch.save(explanation_result, output_dir / f"{base_name}.pt")
    with open(output_dir / f"{base_name}.json", "w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    print(f"\n✅ IG Erklärung gespeichert in: {output_dir}")
    return summary, explanation_result

# IG Explain Button
ig_explain_button = widgets.Button(
    description='IG Erklärung generieren',
    button_style='info',
    icon='search-plus'
)

ig_explain_output = widgets.Output()

def on_ig_explain_click(_b):
    with ig_explain_output:
        clear_output()
        try:
            global ig_explanation_summary, ig_explanation_obj
            ig_explanation_summary, ig_explanation_obj = run_ig_explanation()
            print("\n" + "=" * 50)
            print("IG ERKLÄRUNGSZUSAMMENFASSUNG")
            print("=" * 50)
            print(f"Graph Index: {ig_explanation_summary['graph_idx']}")
            print(f"Node Type: {ig_explanation_summary['node_type']}")
            print(f"Node Index: {ig_explanation_summary['node_idx']}")
            print(f"Vorhersage: {ig_explanation_summary['prediction']:.4f}")
            print(f"Integration Steps: {ig_explanation_summary['n_steps']}")
            print("\nAlle Features (nach Wichtigkeit):")
            feature_imp = ig_explanation_summary['feature_importance']
            sorted_features = sorted(enumerate(feature_imp), key=lambda x: abs(x[1]), reverse=True)
            feature_names = get_feature_names(ig_explanation_summary['node_type'])
            
            for i, (idx, imp) in enumerate(sorted_features):
                name = feature_names[idx] if idx < len(feature_names) else f'Feature_{idx}'
                print(f"  {name}: {imp:.6f}")
            
            print(f"\nTop 5 wichtigste Features:")
            for i, (idx, imp) in enumerate(sorted_features[:5]):
                name = feature_names[idx] if idx < len(feature_names) else f'Feature_{idx}'
                print(f"  {name}: {imp:.6f}")
            
            # Balkendiagramm erstellen
            try:
                plot_single_feature_importance(
                    np.array(feature_imp), 
                    ig_explanation_summary['node_type'], 
                    f" (Graph {ig_explanation_summary['graph_idx']}, Node {ig_explanation_summary['node_idx']})"
                )
            except Exception as e:
                print(f"Fehler bei Diagramm-Erstellung: {e}")


        except Exception as e:
            print(f"\n❌ Fehler bei IG Erklärung: {e}")
            raise

ig_explain_button.on_click(on_ig_explain_click)

display(ig_explain_button)
display(ig_explain_output)


Button(button_style='info', description='IG Erklärung generieren', icon='search-plus', style=ButtonStyle())

Output()

### Batch analysis IG

In [33]:

# IG Batch analysis widgets

ig_batch_model_widget = widgets.Dropdown(

    options=model_files if model_files else ['Keine Modelle gefunden'],

    description='Modell:',

    style={'description_width': 'initial'}

)



ig_batch_data_widget = widgets.Dropdown(

    options=data_files if data_files else ['Keine Daten gefunden'],

    description='Daten:',

    style={'description_width': 'initial'}

)



ig_batch_node_types_widget = widgets.SelectMultiple(

    options=['H', 'C', 'Others'],

    value=['H', 'C'],

    description='Node types:',

    style={'description_width': 'initial'}

)



ig_batch_sample_graphs_widget = widgets.IntText(

    value=0,

    description='Sample graphs (0=all):',

    style={'description_width': 'initial'}

)



ig_batch_n_steps_widget = widgets.IntSlider(

    value=50,

    min=10,

    max=200,

    step=10,

    description='Integration Steps:',

    style={'description_width': 'initial'}

)



ig_batch_baseline_widget = widgets.Dropdown(

    options=['zero', 'mean', 'random', 'min', 'max'],

    value='zero',

    description='Baseline Type:',

    style={'description_width': 'initial'}

)



ig_batch_output_widget = widgets.Text(

    value='results/explanations',

    description='Output Ordner:',

    style={'description_width': 'initial'}

)



# Layout

ig_batch_left = widgets.VBox([

    widgets.HTML('<h4>Daten & Modell</h4>'),

    ig_batch_model_widget,

    ig_batch_data_widget,

    ig_batch_node_types_widget,

    ig_batch_sample_graphs_widget

])



ig_batch_right = widgets.VBox([

    widgets.HTML('<h4>IG Einstellungen</h4>'),

    ig_batch_n_steps_widget,

    ig_batch_baseline_widget,

    ig_batch_output_widget

])



display(widgets.HBox([ig_batch_left, ig_batch_right]))


In [34]:
### 5.2 Run IG Batch Analysis
from scripts.explainer.ig_explainer import batch_explain_nodes_with_ig

from feature_visualization import plot_feature_importance, get_feature_names

import torch



def run_ig_batch_analysis():

    """Run IG batch feature importance analysis across selected graphs and node types."""

    

    # Get dataset size to validate sample_graphs

    data_path = DATA_DIR / ig_batch_data_widget.value

    config_path = MODELS_DIR / 'config.pkl'

    

    if config_path.exists() and data_path.exists():

        from scripts.explainer.explainer_utils import build_dataset, load_config, load_stats

        

        config = load_config(str(config_path))

        norm_stats_path = MODELS_DIR / 'norm_stats.pkl'

        edge_stats_path = MODELS_DIR / 'edge_stats.pkl'

        

        if norm_stats_path.exists() and edge_stats_path.exists():

            norm_stats, edge_stats = load_stats(str(norm_stats_path), str(edge_stats_path))

            dataset = build_dataset(str(data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)

            total_graphs = len(dataset)

        else:

            total_graphs = 1000  # Fallback

    else:

        total_graphs = 1000  # Fallback

    

    # Determine graphs to process

    sample_graphs = ig_batch_sample_graphs_widget.value

    if sample_graphs <= 0 or sample_graphs >= total_graphs:

        graph_indices = list(range(total_graphs))

        print(f"Verarbeite alle {total_graphs} Graphen")

    else:

        graph_indices = list(range(sample_graphs))

        print(f"Verarbeite erste {sample_graphs} von {total_graphs} Graphen")

    

    node_types = list(ig_batch_node_types_widget.value)

    if not node_types:

        print("❌ Fehler: Wähle mindestens einen Node Type aus")

        return

    

    print(f"Node Types: {node_types}")

    print(f"Integration Steps: {ig_batch_n_steps_widget.value}")

    print(f"Baseline: {ig_batch_baseline_widget.value}")

    

    # Prepare paths

    model_path = str(MODELS_DIR / ig_batch_model_widget.value)

    data_path = str(DATA_DIR / ig_batch_data_widget.value)

    config_path = str(MODELS_DIR / 'config.pkl')

    norm_stats_path = str(MODELS_DIR / 'norm_stats.pkl')

    edge_stats_path = str(MODELS_DIR / 'edge_stats.pkl')

    output_dir = str(PROJECT_ROOT / ig_batch_output_widget.value)

    

    # Check if files exist

    required_files = [model_path, data_path, config_path, norm_stats_path, edge_stats_path]

    missing_files = [f for f in required_files if not os.path.exists(f)]

    if missing_files:

        print(f"❌ Fehlende Dateien: {missing_files}")

        return

    

    try:

        # Run batch analysis for each node type

        batch_results = {}

        

        for node_type in node_types:

            print(f"\n=== Verarbeite Node Type: {node_type} ===")

            

            node_batch_results = batch_explain_nodes_with_ig(

                model_path=model_path,

                data_path=data_path,

                config=config_path,

                norm_stats=norm_stats_path,

                edge_stats=edge_stats_path,

                graph_indices=graph_indices,

                node_type=node_type,

                output_dir=output_dir,

                n_steps=ig_batch_n_steps_widget.value,

                baseline_type=ig_batch_baseline_widget.value,

            )

            

            batch_results.update(node_batch_results)

        

        # Display results summary

        print("\n" + "="*60)

        print("IG BATCH-ANALYSE ZUSAMMENFASSUNG")

        print("="*60)

        

        for node_type, result in batch_results.items():

            print(f"\nNode Type: {node_type}")

            print(f"  Analysierte Graphen: {len(graph_indices)}")

            print(f"  Analysierte Knoten: {result['node_count']}")

            print(f"  Features: {len(result['avg_importance'])}")

            print("  Top 5 wichtigste Features:")

            

            # Get feature names

            feature_names = get_feature_names(node_type)

            

            # Sort by absolute importance

            importances = result['avg_importance']

            sorted_indices = sorted(range(len(importances)), key=lambda i: abs(importances[i]), reverse=True)

            

            for i in range(min(5, len(sorted_indices))):

                idx = sorted_indices[i]

                name = feature_names[idx] if idx < len(feature_names) else f'Feature_{idx}'

                print(f"    {name}: {importances[idx]:.6f}")

        

        # Create visualization

        print("\nErstelle Visualisierung...")

        try:

            plot_feature_importance(batch_results)

            print("✅ Visualisierung erfolgreich erstellt!")

        except Exception as e:

            print(f"⚠️ Fehler bei Visualisierung: {e}")

        

        # Save results globally for further analysis

        global ig_batch_results

        ig_batch_results = batch_results

        

        print(f"\n✅ IG Batch-Analyse abgeschlossen!")

        print(f"Ergebnisse gespeichert in: {output_dir}")

        

        return batch_results

        

    except Exception as e:

        print(f"\n❌ Fehler bei IG Batch-Analyse: {e}")

        import traceback

        traceback.print_exc()

        return None



# IG Batch Button

ig_batch_button = widgets.Button(

    description='IG Batch-Analyse starten',

    button_style='success',

    icon='bar-chart'

)



ig_batch_output = widgets.Output()



def on_ig_batch_click(b):

    with ig_batch_output:

        clear_output()

        run_ig_batch_analysis()



ig_batch_button.on_click(on_ig_batch_click)



display(ig_batch_button)

display(ig_batch_output)


Button(button_style='success', description='IG Batch-Analyse starten', icon='bar-chart', style=ButtonStyle())

Output()

In [35]:
# Explainer Widgets
exp_model_widget = widgets.Dropdown(
    options=model_files if model_files else ['Keine Modelle gefunden'],
    description='Modell:',
    style={'description_width': 'initial'}
)

exp_data_widget = widgets.Dropdown(
    options=data_files if data_files else ['Keine Daten gefunden'],
    description='Daten:',
    style={'description_width': 'initial'}
)

exp_graph_idx_widget = widgets.IntText(
    value=0,
    description='Graph Index:',
    style={'description_width': 'initial'}
)

exp_node_type_widget = widgets.Dropdown(
    options=['H', 'C', 'Others'],
    value='H',
    description='Node Type:',
    style={'description_width': 'initial'}
)

exp_node_idx_widget = widgets.IntText(
    value=0,
    description='Node Index:',
    style={'description_width': 'initial'}
)

exp_epochs_widget = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description='Epochen:',
    style={'description_width': 'initial'}
)

exp_lr_widget = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-3,
    max=-1,
    step=0.1,
    description='Learning Rate:',
    style={'description_width': 'initial'},
    readout_format='.3f'
)

exp_type_widget = widgets.Dropdown(
    options=['phenomenon', 'model'],
    value='phenomenon',
    description='Explanation Type:',
    style={'description_width': 'initial'}
)

exp_topk_features_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    description='Top-K Features:',
    style={'description_width': 'initial'}
)

exp_topk_edges_widget = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    description='Top-K Edges:',
    style={'description_width': 'initial'}
)

exp_output_dir_widget = widgets.Text(
    value='results/explanations',
    description='Output Ordner:',
    style={'description_width': 'initial'}
)

# Checkbox für Visualisierung: Alle Nodes erklären
vis_explain_all_nodes_widget = widgets.Checkbox(
    value=True,
    description='Alle Nodes erklären (für Visualisierung)',
    style={'description_width': 'initial'}
)

# Layout
exp_left = widgets.VBox([
    widgets.HTML('<h4>Daten & Modell</h4>'),
    exp_model_widget,
    exp_data_widget,
    exp_graph_idx_widget,
    exp_node_type_widget,
    exp_node_idx_widget
])

exp_right = widgets.VBox([
    widgets.HTML('<h4>Explainer Einstellungen</h4>'),
    exp_epochs_widget,
    exp_lr_widget,
    exp_type_widget,
    exp_topk_features_widget,
    exp_topk_edges_widget,
    exp_output_dir_widget,
    vis_explain_all_nodes_widget
])

display(widgets.HBox([exp_left, exp_right]))

In [36]:
# Generate explanations for ALL nodes with ALL edges (wie in Zelle 19)
from torch_geometric.explain import Explainer, GNNExplainer

# Use graph index selected in Cell 33
graph_idx = int(exp_graph_idx_widget.value)
print(f"Using graph index: {graph_idx}")

# Load selected graph data
data = dataset[graph_idx]

# Build edge_index_dict for the selected graph
edge_index_dict = {}
for store in data.edge_stores:
    key = store._key
    edge_index_dict[key] = store.edge_index

# Build atom_index_dict from NetworkX graph for SDF mapping
nx_g = dataset.nx_graphs[graph_idx]
atom_index_dict = {"H": [], "C": [], "Others": []}
for node in nx_g.nodes():
    element = nx_g.nodes[node]["element"]
    if element in ["H", "C"]:
        atom_index_dict[element].append(node)
    else:
        atom_index_dict["Others"].append(node)

# Convert to tensors for indexing with local indices
for ntype in atom_index_dict:
    if atom_index_dict[ntype]:
        atom_index_dict[ntype] = torch.tensor(atom_index_dict[ntype], dtype=torch.long)
    else:
        atom_index_dict[ntype] = torch.empty(0, dtype=torch.long)

# Prepare feature and edge attribute dicts from selected data
x_dict = {ntype: data[ntype].x for ntype in data.node_types}
edge_attr_dict = {}
for store in data.edge_stores:
    key = store._key
    edge_attr_dict[key] = store.edge_attr

y_dict = {}
for ntype in data.node_types:
    y_val = getattr(data[ntype], 'y', None)
    y_dict[ntype] = y_val

# Load model
in_dim_dict = config.get('in_dim_dict', IN_DIM_DICT)
operator_kwargs = {}
operator_type = config.get('operator_type', 'GATv2Conv')
if operator_type in ['GATConv', 'GATv2Conv']:
    operator_kwargs['add_self_loops'] = False

base_model = HeteroGNNModel(
    in_dim_dict,
    hidden_dim=config.get('hidden_dim', 128),
    out_dim=config.get('out_dim', 1),
    encoder_dropout=config.get('encoder_dropout', 0.0),
    gnnlayer_dropout=config.get('gnnlayer_dropout', 0.0),
    num_gnn_layers=config.get('num_gnn_layers', 3),
    operator_type=operator_type,
    operator_kwargs=operator_kwargs,
    edge_in_dim=10
)
base_model.load_state_dict(torch.load(str(MODELS_DIR / exp_model_widget.value), map_location='cpu'))
base_model.eval()

# Generate explanations for ALL nodes
print("=" * 80)
print("Generiere Erklärungen für ALLE Nodes...")
print("=" * 80)
all_node_summaries = []
node_types_to_explain = ['H', 'C', 'Others']

# Store edge importances per node (SDF index)
edges_per_node = {}

for node_type in node_types_to_explain:
    if node_type not in x_dict:
        continue
    
    num_nodes = x_dict[node_type].size(0)
    print(f"\n{'='*80}")
    print(f"Node Type: {node_type} ({num_nodes} nodes)")
    print(f"{'='*80}")
    
    wrapped_model = NodeTypeRegressionWrapper(base_model, node_type)
    target = None
    target_tensor = y_dict.get(node_type)
    
    if target_tensor is not None and exp_type_widget.value == 'phenomenon':
        target_tensor = target_tensor.reshape(target_tensor.size(0), -1).squeeze(-1)
        target = target_tensor
    
    model_config = ModelConfig(
        mode=ModelMode.regression,
        task_level=ModelTaskLevel.node,
        return_type=ModelReturnType.raw,
    )
    
    explainer = Explainer(
        model=wrapped_model,
        algorithm=GNNExplainer(epochs=exp_epochs_widget.value, lr=exp_lr_widget.value),
        explanation_type=exp_type_widget.value,
        model_config=model_config,
        node_mask_type='attributes',
        edge_mask_type='object',
    )
    
    for node_idx in range(num_nodes):
        try:
            if target is not None and torch.isnan(target[node_idx]):
                continue
            
            explanation = explainer(
                x_dict,
                edge_index_dict,
                edge_attr_dict=edge_attr_dict,
                target=target,
                index=node_idx,
            )
            
            # Collect ALL edges with their importances
            edge_summary = []
            for edge_type, mask in explanation.edge_mask_dict.items():
                if mask is None:
                    continue
                edge_index = edge_index_dict.get(edge_type)
                if edge_index is None:
                    continue
                mask_vals = mask.view(-1).detach().cpu()
                rows = edge_index[0].detach().cpu()
                cols = edge_index[1].detach().cpu()
                for edge_pos, importance in enumerate(mask_vals):
                    edge_summary.append({
                        'edge_type': edge_type,
                        'edge_position': edge_pos,
                        'importance': float(importance),
                        'src_index': int(rows[edge_pos]),
                        'dst_index': int(cols[edge_pos]),
                    })
            
            edge_summary.sort(key=lambda x: abs(x['importance']), reverse=True)
            
            # Find SDF index for this node
            sdf_idx = None
            if atom_index_dict[node_type].numel() > 0:
                for i, idx in enumerate(atom_index_dict[node_type]):
                    if i == node_idx:
                        sdf_idx = int(idx.item())
                        break
            
            if sdf_idx is not None:
                # Convert edge indices to SDF indices
                # Use dict to handle bidirectional edges: keep max importance
                edge_dict = {}
                for edge in edge_summary:
                    edge_type_tuple = edge['edge_type']
                    src_type, _, dst_type = edge_type_tuple
                    src_local = edge['src_index']
                    dst_local = edge['dst_index']
                    
                    if atom_index_dict[src_type].numel() > src_local and atom_index_dict[dst_type].numel() > dst_local:
                        src_sdf = int(atom_index_dict[src_type][src_local].item())
                        dst_sdf = int(atom_index_dict[dst_type][dst_local].item())
                        
                        # Canonical edge key (kleinerer Index zuerst, wie in Zelle 24)
                        edge_key = tuple(sorted([src_sdf, dst_sdf]))
                        
                        # Behalte Maximum bei bidirektionalen Kanten
                        if edge_key in edge_dict:
                            if abs(edge['importance']) > abs(edge_dict[edge_key]['importance']):
                                edge_dict[edge_key] = {
                                    'src': src_sdf,
                                    'dst': dst_sdf,
                                    'importance': edge['importance'],
                                }
                        else:
                            edge_dict[edge_key] = {
                                'src': src_sdf,
                                'dst': dst_sdf,
                                'importance': edge['importance'],
                            }
                
                all_edges_for_node = list(edge_dict.values())
                # Sortiere nach Importance
                all_edges_for_node.sort(key=lambda x: abs(x['importance']), reverse=True)
                
                edges_per_node[sdf_idx] = all_edges_for_node
                
                # Print summary for this node
                symbol = atoms[sdf_idx]['symbol'] if sdf_idx < len(atoms) else '?'
                print(f"\n  Node {sdf_idx} ({symbol}, {node_type}[{node_idx}]): {len(all_edges_for_node)} edges")
                
                # Show ALL edges
                if len(all_edges_for_node) > 0:
                    print(f"    Alle Edges (sortiert nach Importance):")
                    for i, edge in enumerate(all_edges_for_node):
                        src_sym = atoms[edge['src']]['symbol'] if edge['src'] < len(atoms) else '?'
                        dst_sym = atoms[edge['dst']]['symbol'] if edge['dst'] < len(atoms) else '?'
                        print(f"      {i+1:3d}. {edge['src']:2d}({src_sym}) ─ {edge['dst']:2d}({dst_sym}): {edge['importance']:+.6f}")
            
        except Exception as e:
            print(f"  ❌ Fehler bei {node_type}[{node_idx}]: {e}")
            continue

print(f"\n{'='*80}")
print(f"✓ FERTIG!")
print(f"{'='*80}")
print(f"  Erklärungen generiert: {len(edges_per_node)} nodes")
print(f"  Total atoms: {len(atoms)}")
print(f"{'='*80}")

Using graph index: 0


NameError: name 'dataset' is not defined

In [ ]:
def load_sdf_atoms(path):
 
    atoms = []
    with open(path) as f:
        lines = f.readlines()

    if len(lines) < 5:
        return atoms

    counts_line = lines[3]
    try:
        n_atoms = int(counts_line[0:3])
    except ValueError:
        return atoms

    atom_lines = lines[4 : 4 + n_atoms]

    for idx, line in enumerate(atom_lines):
        try:
            x = float(line[0:10])
            y = float(line[10:20])
            z = float(line[20:30])
            symbol = line[31:34].strip()
        except ValueError:
            continue
        atoms.append(
            {
                "index": idx,
                "symbol": symbol,
                "x": x,
                "y": y,
                "z": z,
            }
        )
    return atoms


def calculate_all_atom_importances(nx_g, edge_index_dict, edge_masks, atom_index_dict):
    """
    Compute per-atom importance as the max absolute edge importance over all incident edges.
    Robust to mismatches between edge_index size and mask length by clamping the iteration range.
    """
    atom_importance = {}
    for ntype, atom_indices in atom_index_dict.items():
        if atom_indices.numel() == 0:
            continue
        # Collect edge types touching this node type
        relevant_edge_types = []
        for edge_type in edge_index_dict.keys():
            src_type, _, dst_type = edge_type
            if src_type == ntype or dst_type == ntype:
                relevant_edge_types.append(edge_type)
        # Walk local indices
        for local_idx in range(len(atom_indices)):
            sdf_idx = int(atom_indices[local_idx].item())
            max_importance = 0.0
            for edge_type in relevant_edge_types:
                if edge_type not in edge_masks:
                    continue
                mask = edge_masks[edge_type]
                if mask is None:
                    continue
                # Flatten mask and clamp length to number of edges
                mask = mask.view(-1)
                ei = edge_index_dict[edge_type]
                num_edges = int(ei.size(1))
                num_mask = int(mask.shape[0])
                limit = min(num_edges, num_mask)
                if limit <= 0:
                    continue
                src_type, _, dst_type = edge_type
                # Outgoing
                for edge_idx in range(limit):
                    if ei[0, edge_idx] == local_idx:
                        try:
                            max_importance = max(max_importance, abs(float(mask[edge_idx].item())))
                        except Exception:
                            pass
                # Incoming
                for edge_idx in range(limit):
                    if ei[1, edge_idx] == local_idx:
                        try:
                            max_importance = max(max_importance, abs(float(mask[edge_idx].item())))
                        except Exception:
                            pass
            atom_importance[sdf_idx] = max_importance
    return atom_importance


# Use graph index selected in Cell 33
import json
import torch
import py3Dmol
import pickle
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

graph_idx = int(exp_graph_idx_widget.value)
print(f"Using graph index (Cell 33): {graph_idx}")

data_file = exp_data_widget.value
config_path = MODELS_DIR / "config.pkl"
norm_stats_path = MODELS_DIR / "norm_stats.pkl"
edge_stats_path = MODELS_DIR / "edge_stats.pkl"

with open(config_path, "rb") as f:
    config = pickle.load(f)
with open(norm_stats_path, "rb") as f:
    norm_stats = pickle.load(f)
with open(edge_stats_path, "rb") as f:
    edge_stats = pickle.load(f)

# Load dataset and selected graph
dataset = build_dataset(str(DATA_DIR / data_file), config, norm_stats=norm_stats, edge_stats=edge_stats)

nx_g = dataset.nx_graphs[graph_idx]
data = dataset[graph_idx]

# Build atom index dict (SDF mapping)
atom_index_dict = {"H": [], "C": [], "Others": []}
for node in nx_g.nodes():
    element = nx_g.nodes[node]["element"]
    if element in ["H", "C"]:
        atom_index_dict[element].append(node)
    else:
        atom_index_dict["Others"].append(node)

for ntype in atom_index_dict:
    if atom_index_dict[ntype]:
        atom_index_dict[ntype] = torch.tensor(atom_index_dict[ntype], dtype=torch.long)
    else:
        atom_index_dict[ntype] = torch.empty(0, dtype=torch.long)

# Build edge index dict
edge_index_dict = {}
for store in data.edge_stores:
    key = store._key
    edge_index_dict[key] = store.edge_index

# Gather edges and masks from global explanation if present; else derive from available data
edge_masks = explanation_obj.edge_mask_dict if 'explanation_obj' in globals() and hasattr(explanation_obj, 'edge_mask_dict') else {}

# Compute atom importances (max of incident edges)
atom_importance = calculate_all_atom_importances(nx_g, edge_index_dict, edge_masks, atom_index_dict)

# Prepare SDF
compound = nx_g.graph.get("compound", 1)
structure = nx_g.graph.get("structure", 1)
subfolder = f"{int(compound):03d}"
sdf_name = f"{subfolder}_{int(structure):02d}.sdf"
sdf_path = DATA_DIR / "orca_xyz_formate" / subfolder / sdf_name
print(f"Looking for SDF file: {sdf_path}")

if not sdf_path.exists():
    print("SDF file not found for selected graph.")
else:
    with open(str(sdf_path)) as f:
        sdf_block = f.read()
    atoms = load_sdf_atoms(str(sdf_path))

    # Global edges from masks (if available)
    seen_edges = set()
    all_edges_list = []
    for edge_type in edge_index_dict.keys():
        if edge_type not in edge_masks:
            continue
        ei = edge_index_dict[edge_type]
        mask = edge_masks[edge_type]
        if mask is None:
            continue
        mask = mask.view(-1)
        num_edges = int(ei.size(1))
        num_mask = int(mask.shape[0])
        limit = min(num_edges, num_mask)
        src_type, _, dst_type = edge_type
        for edge_idx in range(limit):
            src_local = int(ei[0, edge_idx].item())
            dst_local = int(ei[1, edge_idx].item())
            if atom_index_dict[src_type].numel() > src_local and atom_index_dict[dst_type].numel() > dst_local:
                src_sdf = int(atom_index_dict[src_type][src_local].item())
                dst_sdf = int(atom_index_dict[dst_type][dst_local].item())
                importance = float(mask[edge_idx].item())
                edge_key = tuple(sorted([src_sdf, dst_sdf])) + (edge_type,)
                if edge_key not in seen_edges:
                    seen_edges.add(edge_key)
                    all_edges_list.append({
                        'src': src_sdf,
                        'dst': dst_sdf,
                        'edge_type': f"{src_type}→{dst_type}",
                        'importance': importance,
                        'edge_idx': edge_idx
                    })
    print(f"Collected {len(all_edges_list)} unique edges from edge_mask_dict for graph {graph_idx}")

    # Build per-node colors using edges_per_node (if present)
    edges_colors_per_node = {}
    if 'edges_per_node' in globals():
        for node_idx, node_edges in edges_per_node.items():
            node_importances = [abs(edge['importance']) for edge in node_edges]
            # Normalizer
            def make_normalizer(values, default=1.0):
                if not values:
                    return lambda v: default
                vmin = min(values); vmax = max(values)
                def norm(v):
                    if vmax == vmin:
                        return default
                    return (v - vmin) / (vmax - vmin)
                return norm
            norm_func = make_normalizer(node_importances)
            # Blue->Red without green
            def importance_to_color_edges(v):
                v = max(0.0, min(1.0, v))
                r = int(v * 255); g = 0; b = int((1.0 - v) * 255)
                return f"rgb({r},{g},{b})"
            colored_edges = []
            for edge in node_edges:
                imp = edge['importance']
                norm_val = norm_func(abs(imp))
                color = importance_to_color_edges(norm_val)
                colored_edges.append({
                    'src': edge['src'],
                    'dst': edge['dst'],
                    'importance': imp,
                    'color': color
                })
            edges_colors_per_node[node_idx] = colored_edges
    edges_colors_per_node_json = json.dumps(edges_colors_per_node)
    print(f"Prepared node-specific edge colors for {len(edges_colors_per_node)} nodes (graph {graph_idx})")

    # Atom colors (optional)
    atom_colors = {}
    if atom_importance:
        vals = list(atom_importance.values())
        def make_atom_norm(vals):
            if not vals:
                return lambda x: 0.0
            m = min(vals); M = max(vals)
            if abs(M - m) < 1e-12:
                return lambda x: 0.0
            return lambda x: (x - m) / (M - m)
        norm_atoms = make_atom_norm(vals)
        def importance_to_color(v):
            if v <= 0:
                return 'blue'
            r = int(255 * v); g = 0; b = int(255 * (1 - v))
            return f'rgb({r},{g},{b})'
        for atom_idx, imp in atom_importance.items():
            v = norm_atoms(imp)
            c = importance_to_color(v)
            atom_colors[str(atom_idx)] = c
    atom_colors_json = json.dumps(atom_colors)

    # Create py3Dmol view
    view = py3Dmol.view(width=700, height=600)
    view.addModel(sdf_block, "sdf")
    view.setStyle({}, {"stick": {"color": "lightgray", "radius": 0.2}, "sphere": {"color": "lightgray", "radius": 0.4}})

    # Labels for non-C atoms
    if atoms:
        for atom in atoms:
            if atom["symbol"] != "C":
                view.addLabel(
                    atom["symbol"],
                    {
                        "position": {"x": atom["x"], "y": atom["y"], "z": atom["z"]},
                        "fontSize": 12,
                        "fontColor": "black",
                        "backgroundColor": "white",
                        "showBackground": True,
                    },
                )

    # Click behavior
    view.addStyle({}, {"clicksphere": {"radius": 0.8}})
    click_cb = f"""
    function(atom, viewer, event, container) {{
        var edgesPerNode = {edges_colors_per_node_json};
        var atomIdx = atom.index;
        if (!viewer.__selectedAtomInfo) {{ viewer.__selectedAtomInfo = {{}}; }}
        viewer.setStyle({{}}, {{stick: {{color: \"lightgray\", radius: 0.2}}, sphere: {{color: \"lightgray\", radius: 0.4}}}});
        viewer.addStyle({{index: atomIdx}}, {{sphere: {{color: \"yellow\", radius: 0.5}}}});
        var nodeEdges = edgesPerNode[atomIdx.toString()];
        if (nodeEdges && nodeEdges.length > 0) {{
            for (var i=0; i<nodeEdges.length; i++) {{
                var edge = nodeEdges[i];
                var src = edge.src; var dst = edge.dst; var color = edge.color;
                viewer.addStyle({{index: src}}, {{stick: {{color: color, radius: 0.3}}}});
                viewer.addStyle({{index: dst}}, {{stick: {{color: color, radius: 0.3}}}});
                if (src !== atomIdx) {{ viewer.addStyle({{index: src}}, {{sphere: {{color: color, radius: 0.45}}}}); }}
                if (dst !== atomIdx) {{ viewer.addStyle({{index: dst}}, {{sphere: {{color: color, radius: 0.45}}}}); }}
            }}
        }}
        if (viewer.__selectedLabel) {{ viewer.removeLabel(viewer.__selectedLabel); }}
        viewer.__selectedLabel = viewer.addLabel(\"Atom \" + atomIdx, {{ position: atom, backgroundColor: \"white\", fontColor: \"black\", fontSize: 12, showBackground: true }});
        viewer.__selectedAtomInfo.index = atomIdx;
        viewer.render();
    }}
    """
    view.setClickable({}, True, click_cb)
    view.zoomTo(); view.render(); view.show()

Using graph index (Cell 33): 140
Looking for SDF file: /Users/sophiaberg/gnn4nmr-4/data/orca_xyz_formate/018/018_00.sdf
Collected 22 unique edges from edge_mask_dict for graph 140
Prepared node-specific edge colors for 9 nodes (graph 140)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.